# FastAPI: a hands-on implementation

**Track:** Production · **Guide:** [`058-fastapi.md`](../guides/concepts/058-fastapi.md)

FastAPI is a Python web framework for defining typed HTTP APIs and automatically documenting their schemas.

> **Intuition:** It is a labeled, validated front desk between callers and application services.

This lesson builds a small working example in stages: prepare inputs, implement
the core idea, run it on a realistic toy case, and inspect its output and limits.
It runs offline unless a cell is explicitly marked as an optional provider call.

```mermaid
flowchart LR
    A[Prepare inputs] --> B[Implement FastAPI]
    B --> C[Run on example]
    C --> D[Evaluate and inspect]
```

**Case study:** Expose a search endpoint with bounded inputs and an explicit timeout/error contract.

Run the cells from top to bottom. To install the notebook environment, follow
[`guides/SETUP.md`](../guides/SETUP.md). The toy data and implementation are for
learning, not a production quality, safety, or security guarantee.


## 1. Define a typed API over a service

FastAPI turns typed Python functions into validated HTTP routes and an OpenAPI schema.


In [ ]:
import asyncio

from fastapi import FastAPI, Query
from httpx import ASGITransport, AsyncClient

app = FastAPI(title="Offline Support Search Demo")
articles = [
    {"id": "a1", "title": "Password reset", "text": "Reset your password from account settings."},
    {"id": "a2", "title": "Returns", "text": "Unused products may be returned within 30 days."},
]

## 2. Implement and validate an endpoint

Bound query length and result count at the API boundary, then keep search logic in a small service function.


In [ ]:
def search_articles(query):
    terms = set(query.lower().split())
    return [
        article for article in articles
        if terms & set((article["title"] + " " + article["text"]).lower().split())
    ]

@app.get("/search")
def search(q: str = Query(min_length=1, max_length=200), limit: int = Query(5, ge=1, le=20)):
    return {"query": q, "results": search_articles(q)[:limit]}

## 3. Exercise HTTP success and validation paths

An in-process ASGI client exercises request parsing without starting a server or opening a port.


In [ ]:
async def exercise_api():
    async with AsyncClient(
        transport=ASGITransport(app=app), base_url="http://test"
    ) as client:
        response = await client.get("/search", params={"q": "password reset"})
        invalid_response = await client.get("/search", params={"q": ""})
    return response, invalid_response

response, invalid_response = asyncio.run(exercise_api())
print("success:", response.status_code, response.json())
print("invalid:", invalid_response.status_code)
assert response.status_code == 200 and invalid_response.status_code == 422

## Wrap-up and next steps

- Identify what was learned, stored, retrieved, or generated in this implementation.
- Change an input and observe how the output changes.
- Test one failure case and decide when the system should abstain or ask for help.
- Replace toy data/components only after you can evaluate the change against a
  held-out or representative test set.

Continue with the [concept guide](../guides/concepts/058-fastapi.md) and the
[learning roadmap](../guides/ROADMAP.md).
